# Data Cleaning - MPB 2025 JO MOnitoring
**Author:** Joshua Nieva (Sr. Process Lead)
**Date:* 03/20/2026
**Source file:** MPB_2025_JO_Monitoring.xlsx

#### Cleaning steps applied:
1. Standardized remarks columns to uppercase
2. Standardized machine name columns to uppercase
3. Stripped date columns to date only
4. Converted % Yield_Cmprsn to float64
5. Cleaned and converted % Yield_Ctng to float64
6. Flagged yield values above 100%
7. Filled null remarks using business logic

In [4]:
# ── Imports ──
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from sqlalchemy import String
from dotenv import load_dotenv
import os

# ── Database Connection ──
load_dotenv()

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

engine = create_engine(f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

# ── Test Connection ──
with engine.connect() as conn:
    print("Connection successful")

file_path = os.getenv("RAW_DATA_PATH")

raw_data = "/Users/toszmac/Documents/LLI-Data_Analysis/LLI-Data-Analysis/data/raw/MPB 2025 JO Monitoring.xlsx"
df = pd.read_excel(file_path)

Connection successful


## EDA checker

In [5]:
#Create a function that performs a brief exploratory data analysis of the raw data to detect all the imperfections of the data

def run_eda(df):
    print("=== SHAPE ===")
    print(df.shape)

    print("\n=== DATA TYPES ===")
    print(df.dtypes)

    print("\n=== MISSING VALUES ===")
    null_counts = df.isna().sum()
    print(null_counts[null_counts > 0])

    print("\n=== UNIQUE VALUE COUNTS (text columns only) ===")
    text_cols = df.select_dtypes(include='str').columns
    for col in text_cols:
        n_unique = df[col].nunique()
        if n_unique < 15:
            print(f"\n{col} ({n_unique} unique values):")
            print(df[col].value_counts(dropna=False))
        else:
            print(f"{col}: {n_unique} unique values — too many to display")

    print("\n=== NUMERIC RANGES ===")
    print(df.describe())
run_eda(df)

=== SHAPE ===
(1064, 28)

=== DATA TYPES ===
No.                                    int64
JO Number                                str
Product Name                             str
Generic                                  str
Dosage Form                              str
Batch Size                             int64
Lot No.                               object
Compounding Remarks                      str
Compounding Date              datetime64[us]
Compounding Wet                          str
Blending Area                            str
Compounding Final Mixing                 str
% Yield_Cmpdg                        float64
Compression Remarks                      str
Compression Date              datetime64[us]
Compression Machine Used                 str
Compression Actual Yield             float64
% Yield_Cmprsn                        object
Encapsulation Remarks                    str
Encapsulation Date            datetime64[us]
Encapsulation Machine Used               str
Encapsulat

In [6]:
#Create a deep copy of the DataFrame
df_clean = df.copy()

##Data Cleaning
#Standardize remarks column by turning all remarks casing into an uppercase
remarks_column = ['Compounding Remarks', 'Compression Remarks', 'Encapsulation Remarks', 'Coating REMARKS']
df_clean[remarks_column] = df_clean[remarks_column].apply(lambda x: x.str.upper())

#Standardize machine names columns to uppercase
machine_column = ['Compression Machine Used', 'Encapsulation Machine Used', 'Coating Machine Used']
df_clean[machine_column] = df_clean[machine_column].apply(lambda x: x.str.upper())

#Strip Date Columns to Date Only
dates_column = ['Compounding Date', 'Compression Date', 'Encapsulation Date', 'Coating Date']
for col in dates_column:
    df_clean[col] = pd.to_datetime(df_clean[col]).dt.normalize()

#Converted % Yield_Cmprsn to float64
df_clean['% Yield_Cmprsn'] = pd.to_numeric(df_clean['% Yield_Cmprsn'], errors='coerce')

#Cleaned and converted % Yield_Ctng to float64
df_clean['% Yield_Ctng'] = df_clean['% Yield_Ctng'].astype(str).replace('nan', pd.NA)
df_clean['% Yield_Ctng'] = pd.to_numeric(
    df_clean['% Yield_Ctng'].str.strip().str.replace(",", ".").str.replace("%", ""),
    errors='coerce'
)

# Normalize values > 1 — these were entered as percentages, not decimals
df_clean.loc[df_clean['% Yield_Ctng'] > 10, '% Yield_Ctng'] = (
    df_clean.loc[df_clean['% Yield_Ctng'] > 10, '% Yield_Ctng'] / 100
)

#Flagged yield values above 100%
df_clean['flag_cmpdg_above_100'] = df_clean['% Yield_Cmpdg']>1.0
df_clean['flag_encap_above_100'] = df_clean['% Yield_Encap']>1.0

## Filled null remarks using business logic

In [7]:

def fill_remarks(row, date_col, not_required_forms=None, not_required_products=None):
    # Rule 1 — date exists means process was done here
    if pd.notna(row[date_col]):
        return "DONE"
    # Rule 2 — dosage form doesn't require this process
    elif not_required_forms and row['Dosage Form'] in not_required_forms:
        return "NOT REQUIRED"
    # Rule 3 — specific product doesn't require this process
    elif not_required_products and row['Product Name'] in not_required_products:
        return "NOT REQUIRED"
    # Rule 4 — process required but no date entry
    else:
        return "PROCESSED OUTSIDE"

# Build encapsulation exclusion list dynamically
all_dosage_forms = df_clean['Dosage Form'].unique().tolist()
not_required_encapsulation = [f for f in all_dosage_forms if f != 'CAPSULE']

not_required_compression = ['CAPSULE']
not_required_coating = ['TABLET', 'CAPSULE', 'BILAYER TABLET']

# Compression
mask = df_clean['Compression Remarks'] != 'DONE'
df_clean.loc[mask, 'Compression Remarks'] = df_clean[mask].apply(
    fill_remarks,
    axis=1,
    date_col='Compression Date',
    not_required_forms=not_required_compression
)

# Encapsulation
mask = df_clean['Encapsulation Remarks'] != 'DONE'
df_clean.loc[mask, 'Encapsulation Remarks'] = df_clean[mask].apply(
    fill_remarks,
    axis=1,
    date_col='Encapsulation Date',
    not_required_forms=not_required_encapsulation
)

# Coating
mask = df_clean['Coating REMARKS'] != 'DONE'
df_clean.loc[mask, 'Coating REMARKS'] = df_clean[mask].apply(
    fill_remarks,
    axis=1,
    date_col='Coating Date',
    not_required_forms=not_required_coating,
    not_required_products=['KALIGEN POTASSIUM CHLORIDE SUSTAINED-RELEASE TABLET 750 MG']
)


## Data Validation
### The script below checks for missing values on critical information needed for each job order

In [8]:
def validate_missing_yields(df_clean):
    
    print("=== DATA VALIDATION REPORT ===\n")
    
    checks = [
        {
            'label': 'Compression yield missing but marked DONE',
            'remarks_col': 'Compression Remarks',
            'yield_col': '% Yield_Cmprsn',
            'date_col': 'Compression Date'
        },
          {
            'label': 'Encapsulation yield missing but marked DONE',
            'remarks_col': 'Encapsulation Remarks',
            'yield_col': '% Yield_Encap',
            'date_col': 'Encapsulation Date'
        },
          {
            'label': 'Coating yield missing but marked DONE',
            'remarks_col': 'Coating REMARKS',
            'yield_col': '% Yield_Ctng',
            'date_col': 'Coating Date'
        }
    ]
#check for missing critical information 
    critical_fields = ['JO Number', 'Product Name', 'Dosage Form', 'Batch Size']
    critical_problems = df_clean[df_clean[critical_fields].isna().any(axis=1)]

    if len(critical_problems) == 0:
        print("✓ Critical fields: No issues found")
    else:
        print(f"✗ Critical fields missing: {len(critical_problems)} rows affected")
        print(critical_problems[['JO Number', 'Product Name', 
                                  'Dosage Form', 'Batch Size']].to_string(index=False))
    
    print()
    
#check for missing values
    for check in checks:
        problem_rows = df_clean[
            (df_clean[check['remarks_col']] == 'DONE') &
            (df_clean[check['yield_col']].isna())
        ]
        
        if len(problem_rows) == 0:
            print(f"✓ {check['label']}: No issues found")
        else:
            print(f"✗ {check['label']}: {len(problem_rows)} rows affected")
            print(problem_rows[['JO Number', 'Product Name', 
                                 check['remarks_col'], 
                                 check['date_col']]].to_string(index=False))
            print()

## Standardize Names

In [9]:
def standardize_generic_names(df):
    corrections = {
        # 1. Atorvastatin salt form inconsistency
        'ATORVASTATIN (AS CALCIUM)': 'ATORVASTATIN CALCIUM',
        
        # 2. Cholecalciferol trailing space
        'CHOLECALCIFEROL (VITAMIN D3) ': 'CHOLECALCIFEROL (VITAMIN D3)',
        
        # 3. Methyldopa non-breaking space
        'METHYLDOPA\xa0(AS SESQUIHYDRATE)': 'METHYLDOPA (AS SESQUIHYDRATE)'
    }
    
    df['Generic'] = df['Generic'].replace(corrections)
    return df

    # Step 2 — Standardize generic names
df_clean = standardize_generic_names(df_clean)

## Handling missing values on numeric columns

In [10]:
def impute_yield(df, yield_col, actual_col, remarks_col):
    
    # Identify rows that need imputation
    mask = (df[remarks_col] == 'DONE') & (df[yield_col].isna())
    
    # Initialize flag column — default is RECORDED
    df[yield_col + '_impute_flag'] = 'RECORDED'
    
    # Option A — derive from actual output and batch size
    can_derive = mask & df[actual_col].notna()
    df.loc[can_derive, yield_col] = df.loc[can_derive, actual_col] / df.loc[can_derive, 'Batch Size']
    df.loc[can_derive, yield_col + '_impute_flag'] = 'DERIVED'
    
    
    # Option B — mean imputation for remaining nulls
    can_impute = mask & df[actual_col].isna()
    recorded_mask = df[yield_col + '_impute_flag'] == 'RECORDED'
    mean_yield = df.loc[recorded_mask, yield_col].mean()
    df.loc[can_impute, yield_col] = mean_yield
    df.loc[can_impute, yield_col + '_impute_flag'] = 'IMPUTED'
    
    return df

In [11]:
def impute_yield(df, yield_col, actual_col, remarks_col):
    
    mask = (df[remarks_col] == 'DONE') & (df[yield_col].isna())
    df[yield_col + '_impute_flag'] = 'RECORDED'
    
    # Option A — derive from actual output and batch size
    can_derive = mask & df[actual_col].notna()
    df.loc[can_derive, yield_col] = (
        df.loc[can_derive, actual_col] / df.loc[can_derive, 'Batch Size']
    )
    df.loc[can_derive, yield_col + '_impute_flag'] = 'DERIVED'
    
    # Option B — smart mean imputation for remaining nulls
    can_impute = mask & df[actual_col].isna()
    
    if can_impute.sum() > 0:
        # Calculate group means
        product_batch_mean = df.groupby(
            ['Product Name', 'Batch Size']
        )[yield_col].transform('mean')

        product_mean = df.groupby(
            ['Product Name']
        )[yield_col].transform('mean')

        global_mean = df[yield_col].mean()

        # Fill in order of priority
        df.loc[can_impute, yield_col] = (
            product_batch_mean[can_impute]
            .fillna(product_mean[can_impute])
            .fillna(global_mean)
        )
        df.loc[can_impute, yield_col + '_impute_flag'] = 'IMPUTED'
    
    return df

## LOAD SCRIPT

In [12]:
def load_dim_product(df):
    dim_product = df[['Product Name', 'Generic', 'Dosage Form']].drop_duplicates()
    
    dim_product = dim_product.rename(columns={
        'Product Name': 'product_name',
        'Generic': 'generic_name',
        'Dosage Form': 'dosage_form'
    })
    
    # Check what's already loaded
    existing = pd.read_sql("SELECT product_name, generic_name, dosage_form FROM dim_product", engine)
    
    # Only keep rows that don't exist yet
    new_rows = dim_product[~dim_product['product_name'].isin(existing['product_name'])]
    
    if len(new_rows) == 0:
        print("dim_product: no new rows to load")
        return dim_product
    
    new_rows.to_sql(
        name='dim_product',
        con=engine,
        if_exists='append',
        index=False,
        dtype={'dosage_form': String}
    )
    
    print(f"dim_product loaded: {len(new_rows)} new rows")
    return dim_product

In [13]:
dim_product_loaded = load_dim_product(df_clean)

dim_product: no new rows to load


In [14]:
def load_dim_machine(df):
    #Wet Granulation
    comp_wet = df[['Compounding Wet']].drop_duplicates().dropna()
    comp_wet = comp_wet.rename(columns={'Compounding Wet': 'machine_name'})
    comp_wet['stage'] = 'compounding'
    comp_wet['machine_type'] = 'granulator'

    #Dry Blending
    comp_dry = df[['Compounding Final Mixing']].drop_duplicates().dropna()
    comp_dry = comp_dry.rename(columns={'Compounding Final Mixing': 'machine_name'})
    comp_dry['stage'] = 'compounding'
    comp_dry['machine_type'] = 'blender' 

    #Compression
    cmprsn = df[['Compression Machine Used']].drop_duplicates().dropna()
    cmprsn = cmprsn.rename(columns={'Compression Machine Used': 'machine_name'})
    cmprsn['stage'] = 'compression'
    cmprsn['machine_type'] = 'tabletting'

    #Encapsulation
    encap = df[['Encapsulation Machine Used']].drop_duplicates().dropna()
    encap = encap.rename(columns={'Encapsulation Machine Used': 'machine_name'})
    encap['stage'] = 'encapsulation'
    encap['machine_type'] = 'encapsulation'

    #Coating
    ctng = df[['Coating Machine Used']].drop_duplicates().dropna()
    ctng = ctng.rename(columns={'Coating Machine Used': 'machine_name'})
    ctng['stage'] = 'coating'
    ctng['machine_type'] = 'coater'

    #Concatenate all into one column 'all_machines'
    dim_machine = pd.concat([comp_wet, comp_dry, cmprsn, encap, ctng])
    dim_machine = dim_machine[~dim_machine['machine_name'].str.contains(
    'c/o|logbook', case=False, na=False
    )]

    #Drop Duplicates
    dim_machine = dim_machine.drop_duplicates(subset=['machine_name'])

    # Check what's already loaded
    existing = pd.read_sql("SELECT machine_name FROM dim_machine", engine)
    
    # Only keep rows that don't exist yet
    new_rows = dim_machine[~dim_machine['machine_name'].isin(existing['machine_name'])]
    
    if len(new_rows) == 0:
        print("dim_machine: no new rows to load")
        return dim_machine
    
    new_rows.to_sql(
        name='dim_machine',
        con=engine,
        if_exists='append',
        index=False,
        dtype={'stage': String}
    )
    
    print(f"dim_machine loaded: {len(new_rows)} new rows")
    return dim_machine

In [15]:
dim_machine_loaded = load_dim_machine(df_clean)

dim_machine: no new rows to load


In [16]:
def load_dim_date():
    date_range = pd.date_range('2023-01-01', '2030-12-31', freq='D')

    records = [
    {
        'date_id': int(d.strftime('%Y%m%d')),
        'full_date': d.date(),
        'year': d.year,
        'quarter': d.quarter,
        'month'        : d.month,
        'month_name'   : d.strftime('%B'),
        'week_number'  : d.isocalendar().week,
        'day_of_week' : d.isoweekday(),              #(1='Monday', 7='Sunday')
        'day_name'    : d.strftime('%A'),
        'is_weekend'  : d.isoweekday() >= 6,
    }
    for d in date_range
    ]
    dim_date = pd.DataFrame(records)

    # Check what's already loaded
    existing = pd.read_sql("SELECT date_id FROM dim_date", engine)
    
    # Only keep rows that don't exist yet
    new_rows = dim_date[~dim_date['date_id'].isin(existing['date_id'])]
    
    if len(new_rows) == 0:
        print("dim_date: no new rows to load")
        return dim_date
    
    new_rows.to_sql(
        name='dim_date',
        con=engine,
        if_exists='append',
        index=False
    )
    
    print(f"dim_date loaded: {len(new_rows)} new rows")
    return dim_date

In [17]:
dim_date_loaded = load_dim_date()

dim_date: no new rows to load


In [18]:
def load_dim_job_order(df):
    # Step 1 — Select columns from df_clean
    dim_job_order = df[['JO Number', 'Product Name', 'Generic',
           'Batch Size', 'Lot No.', 'Dosage Form']].drop_duplicates()

    # Step 2 — Merge to get product_id
    dim_product_db = pd.read_sql("SELECT product_id, product_name, generic_name, dosage_form FROM dim_product", engine)
    dim_job_order = dim_job_order.merge(
    dim_product_db[['product_name', 'generic_name', 'dosage_form', 'product_id']],
    left_on=['Product Name', 'Generic', 'Dosage Form'],
    right_on=['product_name', 'generic_name', 'dosage_form'],
    how='inner'
    )

    # Step 3 — Drop product name columns (keep only product_id)
    dim_job_order = dim_job_order.drop(columns=['Product Name', 'Generic', 'Dosage Form', 
                                  'product_name', 'generic_name', 'dosage_form'])

    # Step 4 — Rename remaining columns to match schema
    dim_job_order = dim_job_order.rename(columns={
        'JO Number': 'jo_number',
        'Batch Size': 'batch_size',
        'Lot No.': 'lot_number'
    })
    
    # Step 5 — Duplicate-safe load
    existing = pd.read_sql("SELECT jo_number FROM dim_job_order", engine)

    new_rows = dim_job_order[~dim_job_order['jo_number'].isin(existing['jo_number'])]

    if len(new_rows) == 0:
        print("dim_job_order: no new rows to load")
        return dim_job_order

    new_rows.to_sql(
        name='dim_job_order',
        con=engine,
        if_exists='append',
        index=False
    )

    print(f"dim_job_order loaded: {len(new_rows)} new rows")
    return dim_job_order


In [19]:
dim_job_order_loaded = load_dim_job_order(df_clean)

dim_job_order: no new rows to load


In [20]:
def load_fact_batch_production(df):

    # ── Read dimension IDs from database ──
    dim_jo_db = pd.read_sql(
        "SELECT jo_id, jo_number, product_id, batch_size FROM dim_job_order", engine)
    dim_machine_db = pd.read_sql(
        "SELECT machine_id, machine_name FROM dim_machine", engine)

    # ── Yield melt ──
    yield_map = {
        '% Yield_Cmpdg':  'compounding',
        '% Yield_Cmprsn': 'compression',
        '% Yield_Ctng':   'coating',
        '% Yield_Encap':  'encapsulation'
    }
    yields_melted = pd.melt(
        df, id_vars=['JO Number'],
        value_vars=list(yield_map.keys()),
        var_name='stage_col', value_name='yield_pct'
    )
    yields_melted['stage'] = yields_melted['stage_col'].map(yield_map)
    yields_melted = yields_melted.drop(columns=['stage_col'])

    # ── Actual output melt ──
    output_map = {
        'Compression Actual Yield':   'compression',
        'Encapsulation Actual Yield': 'encapsulation',
        'Coating Actual Yield':       'coating'
    }
    output_melted = pd.melt(
        df, id_vars=['JO Number'],
        value_vars=list(output_map.keys()),
        var_name='stage_col', value_name='actual_output_units'
    )
    output_melted['stage'] = output_melted['stage_col'].map(output_map)
    output_melted = output_melted.drop(columns=['stage_col'])

    # ── Date melt ──
    date_map = {
        'Compounding Date':    'compounding',
        'Compression Date':    'compression',
        'Encapsulation Date':  'encapsulation',
        'Coating Date':        'coating'
    }
    dates_melted = pd.melt(
        df, id_vars=['JO Number'],
        value_vars=list(date_map.keys()),
        var_name='stage_col', value_name='full_date'
    )
    dates_melted['stage'] = dates_melted['stage_col'].map(date_map)
    dates_melted = dates_melted.drop(columns=['stage_col'])
    dates_melted['date_id'] = pd.to_datetime(
        dates_melted['full_date']
    ).dt.strftime('%Y%m%d').astype('Int64')
    dates_melted = dates_melted.drop(columns=['full_date'])

    # ── Machine melt (compression, encapsulation, coating) ──
    machine_map = {
        'Compression Machine Used':    'compression',
        'Encapsulation Machine Used':  'encapsulation',
        'Coating Machine Used':        'coating'
    }
    machines_melted = pd.melt(
        df, id_vars=['JO Number'],
        value_vars=list(machine_map.keys()),
        var_name='stage_col', value_name='machine_name'
    )
    machines_melted['stage'] = machines_melted['stage_col'].map(machine_map)
    machines_melted = machines_melted.drop(columns=['stage_col'])

    # ── Compounding machines (granulator + blender) ──
    compounding_machines = df[['JO Number', 
                                'Compounding Wet', 
                                'Compounding Final Mixing']].copy()
    compounding_machines['stage'] = 'compounding'
    compounding_machines = compounding_machines.rename(columns={
        'Compounding Wet': 'machine_name',
        'Compounding Final Mixing': 'blending_machine_name'
    })

    # ── Join all melted DataFrames ──
    fact = yields_melted.copy()
    fact = fact.merge(output_melted, on=['JO Number', 'stage'], how='left')
    fact = fact.merge(dates_melted, on=['JO Number', 'stage'], how='left')
    fact = fact.merge(machines_melted, on=['JO Number', 'stage'], how='left')
    fact = fact.merge(
        compounding_machines[['JO Number', 'stage', 
                               'machine_name', 'blending_machine_name']],
        on=['JO Number', 'stage'],
        how='left',
        suffixes=('', '_compounding')
    )

    # ── Consolidate machine columns ──
    fact['machine_name'] = fact['machine_name'].fillna(
        fact['machine_name_compounding']
    )
    fact = fact.drop(columns=['machine_name_compounding'])

    # ── Get FK IDs ──
    fact = fact.merge(dim_jo_db, left_on='JO Number', 
                      right_on='jo_number', how='left')
    fact = fact.merge(
        dim_machine_db.rename(columns={'machine_id': 'machine_id'}),
        on='machine_name', how='left'
    )
    fact = fact.merge(
        dim_machine_db.rename(columns={
            'machine_id': 'blending_machine_id',
            'machine_name': 'blending_machine_name'
        }),
        on='blending_machine_name', how='left'
    )

    # ── Select final columns ──
    fact_final = fact[[
        'jo_id', 'product_id', 'machine_id',
        'blending_machine_id', 'date_id', 'stage',
        'actual_output_units', 'yield_pct', 'batch_size'
    ]].copy()

    # ── Fix data types ──
    int_cols = ['machine_id', 'blending_machine_id', 'date_id', 
            'batch_size', 'actual_output_units']

    for col in int_cols:
        fact_final[col] = pd.to_numeric(fact_final[col], errors='coerce')
        fact_final[col] = fact_final[col].round(0)  # remove any decimal noise
        fact_final[col] = fact_final[col].astype('Int64')

    # ── Duplicate-safe load ──
    existing = pd.read_sql(
        "SELECT jo_id, stage FROM fact_batch_production", engine)
    
    existing['stage'] = existing['stage'].astype(str)
    fact_final['stage'] = fact_final['stage'].astype(str)
    
    existing_keys = set(zip(existing['jo_id'], existing['stage']))
    mask = ~fact_final.apply(
        lambda row: (row['jo_id'], row['stage']) in existing_keys, axis=1
    )
    new_rows = fact_final[mask]

    if len(new_rows) == 0:
        print("fact_batch_production: no new rows to load")
        return fact_final

    new_rows.to_sql(
        name='fact_batch_production',
        con=engine,
        if_exists='append',
        index=False,
        dtype={'stage': String}
    )

    print(f"fact_batch_production loaded: {len(new_rows)} new rows")
    return fact_final

# Run it
fact_loaded = load_fact_batch_production(df_clean)

fact_batch_production: no new rows to load


In [21]:
# Step 1 diagnostic — new cell
comp_dry = (
    df_clean[["Compounding Final Mixing"]].drop_duplicates().dropna()
    .rename(columns={"Compounding Final Mixing": "machine_name"})
)
print(comp_dry["machine_name"].unique())

<StringArray>
['B1 1000L', 'B1 3000L',  'B2 500L', 'B2 3000L',  'B1 500L', 'B2 1000L',
  'B2 150L',  'B1 150L']
Length: 8, dtype: str


In [22]:
# Step 1c — check for name overlap between granulators and blenders
comp_wet = (
    df_clean[["Compounding Wet"]].drop_duplicates().dropna()
    .rename(columns={"Compounding Wet": "machine_name"})
)

overlap = set(comp_dry["machine_name"]) & set(comp_wet["machine_name"])
print("Names appearing in BOTH granulator and blender columns:", overlap)

Names appearing in BOTH granulator and blender columns: set()


In [23]:
# Step 1d — simulate exactly what load_dim_machine() builds before upserting
comp_wet = (
    df_clean[["Compounding Wet"]].drop_duplicates().dropna()
    .rename(columns={"Compounding Wet": "machine_name"})
    .assign(stage="compounding", machine_type="granulator")
)

comp_dry = (
    df_clean[["Compounding Final Mixing"]].drop_duplicates().dropna()
    .rename(columns={"Compounding Final Mixing": "machine_name"})
    .assign(stage="compounding", machine_type="blender")
)

cmprsn = (
    df_clean[["Compression Machine Used"]].drop_duplicates().dropna()
    .rename(columns={"Compression Machine Used": "machine_name"})
    .assign(stage="compression", machine_type="tabletting")
)

encap = (
    df_clean[["Encapsulation Machine Used"]].drop_duplicates().dropna()
    .rename(columns={"Encapsulation Machine Used": "machine_name"})
    .assign(stage="encapsulation", machine_type="encapsulation")
)

ctng = (
    df_clean[["Coating Machine Used"]].drop_duplicates().dropna()
    .rename(columns={"Coating Machine Used": "machine_name"})
    .assign(stage="coating", machine_type="coater")
)

dim_machine = pd.concat([comp_wet, comp_dry, cmprsn, encap, ctng], ignore_index=True)

# Apply the filter
dim_machine = dim_machine[
    ~dim_machine["machine_name"]
    .astype("string")
    .str.contains("c/o|logbook", case=False, na=False)
]

dim_machine = dim_machine.drop_duplicates(subset=["machine_name", "stage", "machine_type"])

print(dim_machine)
print("\nTotal rows:", len(dim_machine))
print("\nBlenders only:")
print(dim_machine[dim_machine["machine_type"] == "blender"])

      machine_name          stage   machine_type
0               G1    compounding     granulator
1               G2    compounding     granulator
3         B1 1000L    compounding        blender
4         B1 3000L    compounding        blender
5          B2 500L    compounding        blender
6         B2 3000L    compounding        blender
7          B1 500L    compounding        blender
8         B2 1000L    compounding        blender
9          B2 150L    compounding        blender
10         B1 150L    compounding        blender
11           ZP-41    compression     tabletting
12        ACCURA D    compression     tabletting
13        ACCURA B    compression     tabletting
14       KARNAVATI    compression     tabletting
15  FLUIDPACK D-27    compression     tabletting
16      PHARMAFILL  encapsulation  encapsulation
17        NJP 3800  encapsulation  encapsulation
18        KEVIN 66        coating         coater
19        KEVIN 48        coating         coater
20            KETO  